# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys
import time

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20_Newsgroups_NOOW.csv',
 '20_Newsgroups_NOOW__internals',
 '20_Newsgroups__internals',
 'MKB_10_NOOW.csv',
 'MKB_10_NOOW__internals',
 'MKB_10__internals',
 'Post_Science_NOOW.csv',
 'Post_Science_NOOW__internals',
 'Post_Science_NOOW_fixed.csv',
 'Post_Science_NOOW_fixed__internals',
 'Post_Science__internals',
 'WikiRef_220_NOOW.csv',
 '_20_Newsgroups.csv',
 '_20_Newsgroups__internals',
 '_Lenta.csv']

In [6]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    # TWENTY_NEWSGROUPS = '20NG_natural_order'

In [7]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'Post_Science_NOOW_fixed.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    # DatasetName.TWENTY_NEWSGROUPS: os.path.join(
    #     DATA_FOLDER_PATH, '20NG_natural_order.csv'
    # ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
}

In [8]:
DATASET_NAME = DatasetName.POSTNAUKA  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [9]:
! head -n 2 $DATASET_FILE_PATH

id,raw_text,vw_text
29998.txt,материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соо

In [10]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join(DATA_FOLDER_PATH, dataset_name.value + '__internals')

In [11]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [12]:
DATASET_INTERNALS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals'

In [13]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 1.32 s, sys: 119 ms, total: 1.43 s
Wall time: 1.35 s


Looking what is inside dataset's folder

In [14]:
DATASET

In [15]:
DATASET.get_possible_modalities()

{'@word'}

In [16]:
MAIN_MODALITY = '@word'

In [17]:
dictionary = DATASET.get_dictionary()

print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=82162)


In [18]:
dictionary

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=82162)

In [19]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=19537)

In [20]:
DATASET._cached_dict = dictionary

In [21]:
DATASET.get_dictionary()

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=19537)

In [22]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['_result', 'result2', 'dict.dict', 'batches', 'vw.txt']

Creating batches

In [23]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/batches", num_batches=4)

In [24]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['_result', 'result2', 'dict.dict', 'batches', 'vw.txt']

In [25]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 3446


Let's look at some text samples

In [26]:
DATASET._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,29998.txt |@word материал отрицательный показа...
7770.txt,7770.txt,культурный код экономика экономист александр а...,7770.txt |@word культурный код экономика эконо...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,32230.txt |@word faq наука третий класс факт э...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,27293.txt |@word обрушение волна поверхность ж...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,481.txt |@word существовать ли суперсимметрия ...


In [27]:
MAIN_MODALITY

'@word'

In [28]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [29]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [30]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 6.43 s, sys: 286 ms, total: 6.72 s
Wall time: 6.64 s


In [31]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [32]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [33]:
ONE_MODEL_NUM_TOPICS = 20
NUM_TOP_WORDS = 20

In [34]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

intra_coherence_score = IntratextCoherenceScore(
    name='toplen_ptw',
    data=DATASET,
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    should_compute=False,
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [35]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [36]:
NUM_ITERATIONS = 20

In [37]:
seed = 0

In [46]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [47]:
DATASET_INTERNALS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals'

In [48]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [49]:
SEARCH_RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result'

In [50]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access '/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result': No such file or directory


In [51]:
BANK_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result/bank__0'

In [52]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [53]:
seed

0

In [37]:
from topnum.model_constructor import init_model_from_family

model = init_model_from_family(
    family='PLSA',
    dataset=DATASET,
    main_modality=MAIN_MODALITY,
    num_topics=10,
    seed=42,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [38]:
model.get_phi().shape

(19186, 10)

In [38]:
from topnum.search_methods.topic_bank.one_model_train_funcs import (default_train_func, _get_topic_model)

In [39]:
model = _get_topic_model(DATASET, num_topics=10)

In [40]:
model.get_phi().shape

(53095, 10)

In [41]:
dictionary = DATASET.get_dictionary()

In [42]:
dictionary

artm.Dictionary(name=475dc0db-1744-492d-8221-1ef62a29a4cf, num_entries=53095)

In [39]:
# https://github.com/machine-intelligence-laboratory/OptimalNumberOfTopics/blob/master/topnum/model_constructor.py#L120C5-L122C73

dictionary = DATASET.get_dictionary()

print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=254bd7ab-7b17-481c-9572-6f2457b051a7, num_entries=53095)


In [40]:
print(dictionary)

artm.Dictionary(name=254bd7ab-7b17-481c-9572-6f2457b051a7, num_entries=19186)


In [41]:
DATASET._cached_dict = dictionary

In [42]:
DATASET.get_dictionary()

artm.Dictionary(name=254bd7ab-7b17-481c-9572-6f2457b051a7, num_entries=19186)

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [54]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = intra_coherence_score,  # coherence_score,
    other_topic_scores = [coherence_score],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [55]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [56]:
optimizer._save_file_path

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result/search_result__0.json'

In [57]:
optimizer._topic_bank._path

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result/bank__0'

Fulfilling the search (get ready for a really long process!):

In [58]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.85s/it]
!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[1.7454794e-09 3.3315755e-09]
 [1.1203075e-08 7.0101908e-10]
 [2.0083948e-10 4.7927644e-09]
 ...
 [4.6382178e-09 5.3519282e-09]
 [1.6459716e-11 5.9757088e-09]
 [3.6669872e-09 2.5503558e-08]].


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 8.3503792e-06]
 [4.4969144e-09 0.0000000e+00 8.3502337e-06]
 [0.0000000e+00 0.0000000e+00 8.3503792e-06]
 ...
 [0.0000000e+00 0.0000000e+00 1.0916711e-05]
 [0.0000000e+00 0.0000000e+00 8.8636452e-06]
 [0.0000000e+00 0.0000000e+00 1.9642241e-05]].
Bank scores: {'perplexity_score': 23124.765625, 'coherence_20': 1.108715281159298, 'diversity_euclidean': 0.06128161394476187, 'diversity_jensenshannon': 0.6049273128020183, 'diversity_hellinger': 0.7027380289919795, 'diversity_cosine': 0.8045688381391027, 'perplexity': 23124.765625, 'ppl_fair': 23124.765625, 'ppl_cheatty': 4137.58447265625}
100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.79s/it]
Creating first level with 2 topics. Dictionary: artm.Di

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 8.3503792e-06]
 [4.4969144e-09 0.0000000e+00 8.3502337e-06]
 [0.0000000e+00 0.0000000e+00 8.3503792e-06]
 ...
 [0.0000000e+00 0.0000000e+00 1.0916711e-05]
 [0.0000000e+00 0.0000000e+00 8.8636452e-06]
 [0.0000000e+00 0.0000000e+00 1.9642241e-05]].
Bank scores: {'perplexity_score': 23124.765625, 'coherence_20': 1.108715281159298, 'diversity_euclidean': 0.061281613944761996, 'diversity_jensenshannon': 0.6049273128025354, 'diversity_hellinger': 0.7027380289921429, 'diversity_cosine': 0.8045688381391178, 'perplexity': 23124.765625, 'ppl_fair': 23124.765625, 'ppl_cheatty': 4137.58447265625}
100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.82s/it]
Creating first level with 2 topics. Dictionary: artm.D

...

!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 8.3503792e-06]
 [4.4969144e-09 0.0000000e+00 8.3502337e-06]
 [0.0000000e+00 0.0000000e+00 8.3503792e-06]
 ...
 [0.0000000e+00 0.0000000e+00 1.0916710e-05]
 [0.0000000e+00 0.0000000e+00 8.8636452e-06]
 [0.0000000e+00 0.0000000e+00 1.9642241e-05]].
Bank scores: {'perplexity_score': 23124.765625, 'coherence_20': 1.108715281159298, 'diversity_euclidean': 0.06128161394476196, 'diversity_jensenshannon': 0.6049273128023781, 'diversity_hellinger': 0.7027380289918433, 'diversity_cosine': 0.8045688381391177, 'perplexity': 23124.765625, 'ppl_fair': 23124.765625, 'ppl_cheatty': 4137.58447265625}
100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.80s/it]
Creating first level with 2 topics. Dictionary: artm.Di

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 8.3503792e-06]
 [4.4969144e-09 0.0000000e+00 8.3502337e-06]
 [0.0000000e+00 0.0000000e+00 8.3503792e-06]
 ...
 [0.0000000e+00 0.0000000e+00 1.0916711e-05]
 [0.0000000e+00 0.0000000e+00 8.8636452e-06]
 [0.0000000e+00 0.0000000e+00 1.9642241e-05]].
Bank scores: {'perplexity_score': 23124.765625, 'coherence_20': 1.108715281159298, 'diversity_euclidean': 0.06128161394476192, 'diversity_jensenshannon': 0.6049273128024786, 'diversity_hellinger': 0.7027380289923382, 'diversity_cosine': 0.804568838139117, 'perplexity': 23124.765625, 'ppl_fair': 23124.765625, 'ppl_cheatty': 4137.58447265625}
100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.80s/it]
Creating first level with 2 topics. Dictionary: artm.Dic

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[8.24879123e-11 0.00000000e+00]
 [6.91189372e-09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 ...
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 2.20617991e-09]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 8.3503792e-06]
 [4.4969144e-09 0.0000000e+00 8.3502337e-06]
 [0.0000000e+00 0.0000000e+00 8.3503792e-06]
 ...
 [0.0000000e+00 0.0000000e+00 1.0916710e-05]
 [0.0000000e+00 0.0000000e+00 8.8636452e-06]
 [0.0000000e+00 0.0000000e+00 1.9642241e-05]].
Bank scores: {'perplexity_score': 23124.765625, 'coherence_20': 1.108715281159298, 'diversity_euclidean': 0.061281613944762, 'diversity_jensenshannon': 0.6049273128024573, 'diversity_hellinger': 0.7027380289922203, 'diversity_cosine': 0.8045688381391178, 'perplexity': 23124.765625, 'ppl_fair': 23124.765625, 'ppl_cheatty': 4137.58447265625}
100%|███████████████████████████████████████████████| 20/20 [16:25<00:00, 49.26s/it]
CPU times: user 28min 27s, sys: 38.7 s, total: 29min 6s
W

In [61]:
len(optimizer._documents)

100

What topics we have in bank

In [59]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1
@word дублирование  8.248791e-11      0.0
      проучиться    6.911894e-09      0.0
      гимнастика    0.000000e+00      0.0
      софист        0.000000e+00      0.0
      парижанин     0.000000e+00      0.0

In [50]:
bank_topics = optimizer._topic_bank.view_topics()

In [51]:
bank_topics.shape

(19186, 8)

In [52]:
bank_topics.head()

topic_0   topic_1   topic_2       topic_3       topic_4  \
@word инвалидность  0.000005  0.000000  0.000000  2.189531e-14  2.561799e-06   
      мазка         0.000011  0.000024  0.000000  0.000000e+00  1.806412e-08   
      professor     0.000000  0.000015  0.000020  6.220338e-14  0.000000e+00   
      умно          0.000000  0.000000  0.000011  2.613295e-16  0.000000e+00   
      игил          0.000000  0.000000  0.000000  0.000000e+00  0.000000e+00   

                         topic_5       topic_6       topic_7  
@word инвалидность  1.603085e-08  0.000000e+00  3.835127e-11  
      мазка         0.000000e+00  0.000000e+00  0.000000e+00  
      professor     0.000000e+00  1.418952e-07  0.000000e+00  
      умно          0.000000e+00  0.000000e+00  0.000000e+00  
      игил          7.317492e-13  1.239376e-13  0.000000e+00

In [53]:
bank_topics['topic_7'].sort_values(ascending=False)[:20]

@word  женщина         0.015624
       мужчина         0.009646
       животное        0.007323
       группа          0.006627
       вид             0.006488
       самец           0.006145
       самка           0.005499
       поведение       0.004960
       большой         0.004550
       сон             0.004252
       эволюция        0.004244
       признак         0.003649
       обезьяна        0.003567
       жить            0.003537
       ген             0.003508
       ребёнок         0.003389
       шимпанзе        0.003304
       некоторый       0.003303
       неандерталец    0.003157
       раса            0.003151
Name: topic_7, dtype: float64

And topic scores

In [54]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7
kernel_size,2856.000000,2425.000000,2598.000000,3092.000000,3122.000000,3179.000000,3089.000000,3010.000000
coherence_20,1.181293,0.940944,0.927175,0.858446,0.960241,0.942290,1.144227,0.891899
distance_to_nearest,0.790259,0.661715,0.709441,0.793903,0.678832,0.795731,0.782820,0.770218


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [55]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [56]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [57]:
optimizer._result['num_bank_topics']

[2, 3, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 8, 8, 8]

In [58]:
len(optimizer._result['bank_topic_scores'])

20

In [59]:
optimizer._result

{'optimum': 8,
 'optimum_std': 5.0,
 'bank_scores': [{'perplexity_score': 27600.54296875,
   'coherence_20': 1.1177666958123678,
   'diversity_euclidean': 0.04971374671297815,
   'diversity_jensenshannon': 0.5967539527734701,
   'diversity_hellinger': 0.6922578888429283,
   'diversity_cosine': 0.746649317119927,
   'perplexity': 27600.54296875,
   'ppl_fair': 27600.54296875,
   'ppl_cheatty': 4970.97119140625},
  {'perplexity_score': 12274.9267578125,
   'coherence_20': 1.0511303118773,
   'diversity_euclidean': 0.045677036265274085,
   'diversity_jensenshannon': 0.6265346154678154,
   'diversity_hellinger': 0.7277956636084046,
   'diversity_cosine': 0.763120532644456,
   'perplexity': 12274.9267578125,
   'ppl_fair': 12274.9267578125,
   'ppl_cheatty': 4744.744140625},
  {'perplexity_score': 10446.2529296875,
   'coherence_20': 0.9918173215187528,
   'diversity_euclidean': 0.046234306391336584,
   'diversity_jensenshannon': 0.6043805692629273,
   'diversity_hellinger': 0.6993633250185

In [60]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 2540,
   'coherence_20': 1.054240465084224,
   'distance_to_nearest': 0.0},
  {'kernel_size': 2856,
   'coherence_20': 1.1812929265405114,
   'distance_to_nearest': 0.79025919077267}],
 [{'kernel_size': 2540,
   'coherence_20': 1.054240465084224,
   'distance_to_nearest': 0.0},
  {'kernel_size': 2856,
   'coherence_20': 1.1812929265405114,
   'distance_to_nearest': 0.79025919077267},
  {'kernel_size': 3455,
   'coherence_20': 0.9178575440071646,
   'distance_to_nearest': 0.8382437670066345}],
 [{'kernel_size': 2856,
   'coherence_20': 1.1812929265405114,
   'distance_to_nearest': 0.79025919077267},
  {'kernel_size': 3455,
   'coherence_20': 0.9178575440071646,
   'distance_to_nearest': 0.8382437670066345},
  {'kernel_size': 2425,
   'coherence_20': 0.9409436630681554,
   'distance_to_nearest': 0.6617147426494687},
  {'kernel_size': 2598,
   'coherence_20': 0.9271751524591798,
   'distance_to_nearest': 0.7094405645172381}],
 [{'kernel_size': 2856,
   'coherence_20': 1.

In [61]:
optimizer._result['model_scores'][0]

{'perplexity_score': 2983.59326171875,
 'coherence_20': 0.705566582121322,
 'diversity_euclidean': 0.04656994985355387,
 'diversity_jensenshannon': 0.5982341411716676,
 'diversity_hellinger': 0.6850917867989588,
 'diversity_cosine': 0.7077267888360923,
 'perplexity': 2983.59326171875}

In [62]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 8

0.9808143276550801

In [63]:
len(optimizer._result['bank_scores'])

20

In [64]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 6107.216796875,
 'coherence_20': 0.98081432765508,
 'diversity_euclidean': 0.04722042866291265,
 'diversity_jensenshannon': 0.6019874723801198,
 'diversity_hellinger': 0.6929206053597949,
 'diversity_cosine': 0.7453124228974252,
 'perplexity': 6107.216796875,
 'ppl_fair': 6107.216796875,
 'ppl_cheatty': 4218.52294921875}

In [133]:
# Real-fixing bank topics

In [85]:
from topnum.search_methods.topic_bank.one_model_train_funcs import _get_topic_model

In [86]:
model

Model(id=--12h57m59s_22d03m2024y---, parent_id=None, experiment_id=None)

In [115]:
phi = model.get_phi()
phi = phi.iloc[phi.index.get_level_values(0).isin([MAIN_MODALITY])]

word2index = {
    word: index for index, word in enumerate(phi.index)
}

bank_phi = optimizer._get_phi(optimizer._topic_bank.topics, word2index)
bank_model = _get_topic_model(
    optimizer._dataset,
    phi=bank_phi,
    scores=optimizer._all_model_scores,
    num_safe_fit_iterations=1
)

In [126]:
bank_phi['topic_7'].sort_values(ascending=False)

@word  женщина        0.015624
       мужчина        0.009646
       животное       0.007323
       группа         0.006627
       вид            0.006488
                        ...   
       косить         0.000000
       отдалять       0.000000
       натянутый      0.000000
       драматичный    0.000000
       аврелий        0.000000
Name: topic_7, Length: 19186, dtype: float64

In [122]:
bank_topics['topic_7'].sort_values(ascending=False)

@word  женщина        0.015624
       мужчина        0.009646
       животное       0.007323
       группа         0.006627
       вид            0.006488
                        ...   
       косить         0.000000
       отдалять       0.000000
       натянутый      0.000000
       драматичный    0.000000
       аврелий        0.000000
Name: topic_7, Length: 19186, dtype: float64

In [123]:
bank_model.get_phi()['topic_7'].sort_values(ascending=False)

modality  token      
@word     ребёнок        0.008598
          группа         0.006715
          вид            0.006035
          женщина        0.005993
          мозг           0.005540
                           ...   
          схватывание    0.000000
          немецк         0.000000
          предвзятый     0.000000
          умник          0.000000
          аврелий        0.000000
Name: topic_7, Length: 19186, dtype: float32

In [125]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, phi, topic_names: List[str]):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._phi = phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)
        parent_phi = self._phi
        
        rwt[:, self._topic_indices] += parent_phi.values[:, self._topic_indices]

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [127]:
regularizer = FastFixPhiRegularizer(
    name='fix',
    phi=bank_phi,
    topic_names=bank_phi.columns,
)

In [128]:
bank_model._fit(
    optimizer._dataset.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

In [129]:
bank_model.get_phi()['topic_7'].sort_values(ascending=False)

modality  token      
@word     женщина        0.015622
          мужчина        0.009645
          животное       0.007322
          группа         0.006628
          вид            0.006488
                           ...   
          полимер        0.000000
          разлагаться    0.000000
          микросхема     0.000000
          принтер        0.000000
          аврелий        0.000000
Name: topic_7, Length: 19186, dtype: float32

In [131]:
coherence_score.call_by_topic(bank_model)

{0: array([1.18129293]),
 1: array([0.94094366]),
 2: array([0.92717515]),
 3: array([0.8584458]),
 4: array([0.96024055]),
 5: array([0.94228977]),
 6: array([1.14422727]),
 7: array([0.89189949])}

In [132]:
coherence_score.call(bank_model)

array([0.98081433])

In [135]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 3875.74169921875,
 'coherence_20': 0.564805432002065,
 'diversity_euclidean': 0.03567965163581824,
 'diversity_jensenshannon': 0.5957037221303795,
 'diversity_hellinger': 0.6853192252936954,
 'diversity_cosine': 0.6242113090660045,
 'perplexity': 3875.74169921875}

In [136]:
optimizer._get_default_scores(bank_model)

{'perplexity_score': 5749.072265625,
 'coherence_20': 0.98081432765508,
 'diversity_euclidean': 0.04722044093307479,
 'diversity_jensenshannon': 0.6019876723935436,
 'diversity_hellinger': 0.6929207704718845,
 'diversity_cosine': 0.745312495094132,
 'perplexity': 5749.072265625}

In [78]:
model = init_model_from_family('ARTM', DATASET, MAIN_MODALITY, 2, 0)

No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [80]:
m._fit(DATASET.get_batch_vectorizer(), 2)

In [82]:
m.get_phi()

topic_0   topic_1  background_2
modality token                                         
@word    инвалидность  0.000000  0.000000      0.000011
         мазка         0.000000  0.000000      0.000011
         professor     0.000000  0.000000      0.000011
         умно          0.000000  0.000000      0.000011
         игил          0.000000  0.000000      0.000011
...                         ...       ...           ...
         милосердие    0.000000  0.000000      0.000015
         поверка       0.000000  0.000000      0.000015
         вто           0.000083  0.000000      0.000025
         слоить        0.000003  0.000012      0.000076
         акт           0.000210  0.000083      0.000123

[19186 rows x 3 columns]

In [38]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [39]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [40]:
NUM_ITERATIONS = 20

In [41]:
seed = 0

In [42]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [43]:
DATASET_INTERNALS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals'

In [44]:
! ls $DATASET_INTERNALS_FOLDER_PATH

batches  dict.dict  _result  result2  vw.txt


In [45]:
! mv $DATASET_INTERNALS_FOLDER_PATH/result $DATASET_INTERNALS_FOLDER_PATH/_result

mv: cannot stat '/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result': No such file or directory


In [46]:
! ls $DATASET_INTERNALS_FOLDER_PATH

batches  dict.dict  _result  result2  vw.txt


In [47]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [48]:
SEARCH_RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result2'

In [49]:
! ls /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals

batches  dict.dict  _result  result2  vw.txt


In [50]:
! mv /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result2 /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/_result2

In [51]:
! ls /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals

batches  dict.dict  _result  _result2  vw.txt


In [67]:
! ls $SEARCH_RESULTS_FOLDER_PATH

bank__0  search_result__0.json


In [52]:
BANK_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result2/bank__0'

In [69]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [54]:
seed

0

In [39]:
# https://github.com/machine-intelligence-laboratory/OptimalNumberOfTopics/blob/master/topnum/model_constructor.py#L120C5-L122C73

dictionary = DATASET.get_dictionary()

print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=d94d49a7-9d23-4f72-b02b-d89dfec6fdee, num_entries=53095)


In [40]:
print(dictionary)

artm.Dictionary(name=d94d49a7-9d23-4f72-b02b-d89dfec6fdee, num_entries=19186)


In [41]:
DATASET._cached_dict = dictionary

In [55]:
DATASET.get_dictionary()

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=19537)

In [56]:
DATASET.get_dictionary()

artm.Dictionary(name=2901dd17-5ad8-409a-bf5d-782758d4d009, num_entries=19537)

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [57]:
len(DOCUMENTS), DATASET._data.shape

(3446, (3446, 3))

In [58]:
ONE_MODEL_NUM_TOPICS

20

In [59]:
NUM_ITERATIONS

20

In [70]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = intra_coherence_score,  # coherence_score,
    other_topic_scores = [coherence_score],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
    documents          = DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)
    #                                  70%                    75%                   80%
    topic_score_threshold_percentile = 2.3394925682940233,  # 2.435017475164776,  # 2.5432101128006273,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:218: UserWarning: topic_score_threshold_percentile 2.3394925682940233 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [71]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [72]:
optimizer._save_file_path

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result2/search_result__0.json'

In [73]:
optimizer._topic_bank._path

'/data_mil/shared/CompressaAI/iterative/data/noow/Post_Science__internals/result2/bank__0'

Fulfilling the search (get ready for a really long process!):

In [74]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

 50%|████████████████████████                        | 1/2 [01:42<01:42, 102.67s/it]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 2/2 [01:42<00:00, 51.42s/it]
Using absoulte threshold: 2.3394925682940233.
Eliminating bcg topic before Hier. Cur |T| is 21, topics are: Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'background_20'],
      dtype='object').
Now |T| is 20, topics are: Inde

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 ...
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 8.6239379e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 8.6239379e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 8.6239379e-06]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 1.1274344e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 9.1540196e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 2.0285726e-05]].
Bank scores: {'perplexity_score': 28439.177734375, 'coherence_20': 0.9567557076597479, 'diversity_euclidean': 0.07520688316023542, 'diversity_jensenshannon': 0.6518311789823222, 'diversity_hellinger': 0.7552062776689774, 'diversity_cosine': 0.8216776639936949, 'perplexity': 28439.177734375, 'ppl_fair': 28439.177734375, 'ppl_cheatty': 4144.4482421875}
  5%|██▎                                            | 1/20 [02:35<49:10, 155.29s/it]No spec topics
No spec topics


...

!!! Bank Phi: [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.3065614e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.0608412e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 2.3508723e-05]].
Bank scores: {'perplexity_score': 16017.2265625, 'coherence_20': 0.7282088768668022, 'diversity_euclidean': 0.0641380560766113, 'diversity_jensenshannon': 0.6174077723197019, 'diversity_hellinger': 0.7146702985426278, 'diversity_cosine': 0.734793917042038, 'perplexity

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

 50%|████████████████████████                        | 1/2 [01:44<01:44, 104.62s/it]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 2/2 [01:44<00:00, 52.39s/it]
Using absoulte threshold: 2.3394925682940233.
Eliminating bcg topic before Hier. Cur |T| is 21, topics are: Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'background_20'],
      dtype='object').
Now |T| is 20, topics are: Inde

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 9.9941117e-06]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.3065614e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.0608412e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 2.3508723e-05]].
Bank scores: {'perplexity_score': 16017.2265625, 'coherence_20': 0.7282088768668022, 'diversity_euclidean': 0.06413805607661366, 'diversity_jensenshannon': 0.6174077723201977, 'diversity_hellinger': 0.7146702985437808, 'diversity_cosine': 0.7347939170421168, 'perplexi

In [75]:
optimizer._main_modality

'@word'

What topics we have in bank

In [76]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5
@word дублирование      0.0      0.0      0.0      0.0      0.0      0.0
      проучиться        0.0      0.0      0.0      0.0      0.0      0.0
      гимнастика        0.0      0.0      0.0      0.0      0.0      0.0
      софист            0.0      0.0      0.0      0.0      0.0      0.0
      парижанин         0.0      0.0      0.0      0.0      0.0      0.0

In [77]:
bank_topics = optimizer._topic_bank.view_topics()

In [78]:
bank_topics.shape

(19537, 6)

In [79]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5
@word дублирование      0.0      0.0      0.0      0.0      0.0      0.0
      проучиться        0.0      0.0      0.0      0.0      0.0      0.0
      гимнастика        0.0      0.0      0.0      0.0      0.0      0.0
      софист            0.0      0.0      0.0      0.0      0.0      0.0
      парижанин         0.0      0.0      0.0      0.0      0.0      0.0

In [86]:
bank_topics['topic_1'].sort_values(ascending=False)[:20]  # TODO: Всё равно какая-то "полуфоновая" тема попала в банк

@word  говорить      0.013766
       там           0.011863
       понимать      0.011178
       знать         0.011130
       сейчас        0.010280
       дело          0.009853
       конечно       0.009372
       делать        0.009064
       надо          0.008819
       должный       0.008294
       чтото         0.007880
       вещь          0.007748
       вообще        0.007538
       заниматься    0.007420
       хорошо        0.006877
       разный        0.006778
       сделать       0.006634
       какието       0.006623
       даже          0.006439
       хотеть        0.006165
Name: topic_1, dtype: float64

And topic scores

In [87]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5
kernel_size,1683.000000,1591.000000,1655.000000,1833.000000,1981.000000,1964.000000
toplen_ptw,2.390119,3.234490,2.531433,2.418970,3.008297,3.044554
coherence_20,1.180297,0.439105,0.442914,0.636223,0.931885,0.738829
distance_to_nearest,0.837881,0.852757,0.797195,0.732147,0.693357,0.792311


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [89]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [90]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [91]:
optimizer._result['num_bank_topics']

[3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6]

In [92]:
len(optimizer._result['bank_topic_scores'])

20

In [93]:
optimizer._result

{'optimum': 6,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 28439.177734375,
   'coherence_20': 0.9567557076597479,
   'diversity_euclidean': 0.07520688316023542,
   'diversity_jensenshannon': 0.6518311789823222,
   'diversity_hellinger': 0.7552062776689774,
   'diversity_cosine': 0.8216776639936949,
   'perplexity': 28439.177734375,
   'ppl_fair': 28439.177734375,
   'ppl_cheatty': 4144.4482421875},
  {'perplexity_score': 28439.177734375,
   'coherence_20': 0.9567557076597479,
   'diversity_euclidean': 0.07520688316023438,
   'diversity_jensenshannon': 0.6518311789829264,
   'diversity_hellinger': 0.7552062776686018,
   'diversity_cosine': 0.8216776639937158,
   'perplexity': 28439.177734375,
   'ppl_fair': 28439.177734375,
   'ppl_cheatty': 4144.4482421875},
  {'perplexity_score': 28439.177734375,
   'coherence_20': 0.9567557076597479,
   'diversity_euclidean': 0.07520688316023437,
   'diversity_jensenshannon': 0.6518311789822113,
   'diversity_hellinger': 0.75520627766

In [94]:
len(optimizer._result['bank_topic_scores'])

20

In [95]:
[len(s) for s in optimizer._result['bank_topic_scores']]

[3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6]

In [96]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 1676,
   'toplen_ptw': 2.9137328999468073,
   'coherence_20': 1.2508650222621893,
   'distance_to_nearest': 0.0},
  {'kernel_size': 1683,
   'toplen_ptw': 2.390118941026358,
   'coherence_20': 1.180296731842603,
   'distance_to_nearest': 0.8378806766600361},
  {'kernel_size': 1591,
   'toplen_ptw': 3.234490197330379,
   'coherence_20': 0.43910536887445184,
   'distance_to_nearest': 0.8527573519638529}],
 [{'kernel_size': 1676,
   'toplen_ptw': 2.9137328999468073,
   'coherence_20': 1.2508650222621893,
   'distance_to_nearest': 0.0},
  {'kernel_size': 1683,
   'toplen_ptw': 2.390118941026358,
   'coherence_20': 1.180296731842603,
   'distance_to_nearest': 0.8378806766600361},
  {'kernel_size': 1591,
   'toplen_ptw': 3.234490197330379,
   'coherence_20': 0.43910536887445184,
   'distance_to_nearest': 0.8527573519638529}],
 [{'kernel_size': 1676,
   'toplen_ptw': 2.9137328999468073,
   'coherence_20': 1.2508650222621893,
   'distance_to_nearest': 0.0},
  {'kernel_size': 

In [97]:
optimizer._result['model_scores'][0]

{'perplexity_score': 2774.172119140625,
 'coherence_20': 0.7917501820424284,
 'diversity_euclidean': 0.0697766256356419,
 'diversity_jensenshannon': 0.6568446692721249,
 'diversity_hellinger': 0.7646332792232048,
 'diversity_cosine': 0.811718425985906,
 'perplexity': 2774.172119140625,
 'toplen_ptw': 2.0139480500213662}

In [98]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 14

0.3120895186572009

In [99]:
len(optimizer._result['bank_scores'])

20

In [100]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 16017.2265625,
 'coherence_20': 0.7282088768668022,
 'diversity_euclidean': 0.06413805607661366,
 'diversity_jensenshannon': 0.6174077723201977,
 'diversity_hellinger': 0.7146702985437808,
 'diversity_cosine': 0.7347939170421168,
 'perplexity': 16017.2265625,
 'ppl_fair': 16017.2265625,
 'ppl_cheatty': 3733.1591796875}

In [101]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 1994,
   'toplen_ptw': 2.0103522136784897,
   'coherence_20': 0.5030885868011427},
  {'kernel_size': 2247,
   'toplen_ptw': 2.1518301958752915,
   'coherence_20': 0.5542258711717113},
  {'kernel_size': 1727,
   'toplen_ptw': 2.111886926248693,
   'coherence_20': 0.46083629281021266},
  {'kernel_size': 2437,
   'toplen_ptw': 1.9120912737809934,
   'coherence_20': 0.7784617214308792},
  {'kernel_size': 2092,
   'toplen_ptw': 2.053203908557405,
   'coherence_20': 0.6874723289012623},
  {'kernel_size': 1676,
   'toplen_ptw': 2.9137328999468073,
   'coherence_20': 1.2508650222621893,
   'distance_to_nearest': 0.0},
  {'kernel_size': 1924,
   'toplen_ptw': 1.7453635506232241,
   'coherence_20': 0.765747875827312},
  {'kernel_size': 1916,
   'toplen_ptw': 1.6741463062821191,
   'coherence_20': 0.7355746840071614},
  {'kernel_size': 2123,
   'toplen_ptw': 2.0139480500213662,
   'coherence_20': 0.8050386426539775},
  {'kernel_size': 2528,
   'toplen_ptw': 1.877373249189409,
  

In [133]:
# Real-fixing bank topics

In [225]:
from topnum.search_methods.topic_bank.one_model_train_funcs import _get_topic_model, init_phi_utils

In [574]:
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def _get_topic_model_dev(
        dataset: Dataset,
        phi: pd.DataFrame = None,
        num_topics: int = None,
        seed: int = None,
        scores: List = None,
        num_safe_fit_iterations: int = 3,
        num_processors: int = 3,
        cache_theta: bool = False) -> TopicModel:

    dictionary = dataset.get_dictionary()

    # for modality in dataset.get_possible_modalities():
    #     if modality not in modalities_to_use:
    #         dictionary.filter(class_id=modality, max_df=0, inplace=True)

    if num_topics is not None and phi is not None:
        assert num_topics >= phi.shape[1]
    elif num_topics is None and phi is not None:
        num_topics = phi.shape[1]
    elif num_topics is None and phi is None:
        raise ValueError()

    topic_names = [f'topic_{i}' for i in range(num_topics)]

    # if seed is None:
    #     artm_model = artm.ARTM(topic_names=topic_names)
    # else:
    #     artm_model = artm.ARTM(topic_names=topic_names, seed=seed)

    if seed is None:
        artm_model = artm.ARTM(topic_names=topic_names, class_ids={MAIN_MODALITY: 1})  # TODO: not list, but dict!!!
    else:
        artm_model = artm.ARTM(topic_names=topic_names, seed=seed, class_ids={MAIN_MODALITY: 1})

    # artm_model = init_model(topic_names, class_ids=[MAIN_MODALITY])

    # artm_model = init_plsa(DATASET, [MAIN_MODALITY], MAIN_MODALITY, 5)

    artm_model.num_processors = num_processors
    artm_model.initialize(dictionary)

    """
    if phi is None:
        pass
    elif num_safe_fit_iterations is not None and num_safe_fit_iterations > 0:
        init_phi_utils._safe_copy_phi(artm_model, phi, dataset, num_safe_fit_iterations)
    else:
        init_phi_utils._copy_phi(artm_model, phi)
    """
    # this breaks smth in ARTM
    # test_ppl@word [1827.4515380859375, 2707.63623046875, 2707.67919921875, 2707.679443359375, 2707.679443359375]
    # test_ppl@word_with_d [4073.36328125, 6035.2822265625, 6035.3779296875, 6035.37841796875, 6035.37841796875]
    # test_ppl@all [1827.4515380859375, 2707.63623046875, 2707.67919921875, 2707.679443359375, 2707.679443359375]
    # test_ppl@all_2 [1827.4515380859375, 2707.63623046875, 2707.67919921875, 2707.679443359375, 2707.679443359375]
    # test_ppl@all_2_with_d [4073.36328125, 6035.2822265625, 6035.3779296875, 6035.37841796875, 6035.37841796875]
    
    topic_model = TopicModel(
        artm_model=artm_model,
        model_id='0',
        cache_theta=cache_theta,
        theta_columns_naming='title'
    )

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    return topic_model

In [227]:
model = artm_train_func(DATASET, 1, 2, 1)

No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [424]:
DATASET.get_dictionary()

artm.Dictionary(name=6231af29-73db-42d7-a978-1e0c744be737, num_entries=19186)

In [425]:
optimizer._dataset.get_dictionary()

artm.Dictionary(name=6231af29-73db-42d7-a978-1e0c744be737, num_entries=19186)

In [588]:
phi = model.get_phi()
phi = phi.iloc[phi.index.get_level_values(0).isin([MAIN_MODALITY])]

word2index = {
    word: index for index, word in enumerate(phi.index)
}

bank_phi = optimizer._get_phi(optimizer._topic_bank.topics, word2index)
# bank_model = _get_topic_model_dev(  #_get_topic_model( _get_topic_model_dev
#     optimizer._dataset,
#     phi=bank_phi,
#     scores=optimizer._all_model_scores,
#     num_safe_fit_iterations=1,
# )
bank_model = init_model_from_family('sparse', DATASET, MAIN_MODALITY, 5, 0)


"""
_bank_model = init_plsa(DATASET, [MAIN_MODALITY], MAIN_MODALITY, 5)
_bank_model.num_processors = 3
# _bank_model.seed = seed
dictionary = DATASET.get_dictionary()

# TODO: maybe this cycle is not necessary
# for modality in DATASET.get_possible_modalities():
#     if modality not in [MAIN_MODALITY]:
#         dictionary.filter(class_id=modality, max_df=0, inplace=True)

_bank_model.initialize(dictionary)
# add_standard_scores(model, dictionary, main_modality=main_modality,
#                     all_modalities=modalities_to_use)

bank_model = TopicModel(
    artm_model=_bank_model,
    # custom_regularizers=custom_regs
)
"""

No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


'\n_bank_model = init_plsa(DATASET, [MAIN_MODALITY], MAIN_MODALITY, 5)\n_bank_model.num_processors = 3\n# _bank_model.seed = seed\ndictionary = DATASET.get_dictionary()\n\n# TODO: maybe this cycle is not necessary\n# for modality in DATASET.get_possible_modalities():\n#     if modality not in [MAIN_MODALITY]:\n#         dictionary.filter(class_id=modality, max_df=0, inplace=True)\n\n_bank_model.initialize(dictionary)\n# add_standard_scores(model, dictionary, main_modality=main_modality,\n#                     all_modalities=modalities_to_use)\n\nbank_model = TopicModel(\n    artm_model=_bank_model,\n    # custom_regularizers=custom_regs\n)\n'

In [589]:
bank_model._model.class_ids

{'@word': 1}

In [590]:
bank_model._model.regularizers

[smooth_phi_bcg, smooth_theta_bcg, sparse_phi_sp, sparse_theta_sp]

In [591]:
bank_model._fit(
    optimizer._dataset.get_batch_vectorizer(),
    num_iterations=2,
)

In [592]:
for m in DATASET.get_possible_modalities():
    bank_model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}',
            class_ids=[m]
        )
    )
    bank_model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}_with_d',
            class_ids=[m],
            dictionary=DATASET.get_dictionary()
        )
    )

bank_model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all',
        class_ids=list(DATASET.get_possible_modalities()),
    )
)
bank_model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2',
   )
)
bank_model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2_with_d',
        dictionary=DATASET.get_dictionary()
   )
)

In [593]:
bank_phi['topic_4'].sort_values(ascending=False)

@word  клетка             0.030503
       ген                0.012852
       организм           0.009684
       днк                0.009340
       пациент            0.009092
                            ...   
       камень             0.000000
       встревожить        0.000000
       технологический    0.000000
       летопись           0.000000
       аврелий            0.000000
Name: topic_4, Length: 19186, dtype: float64

In [594]:
bank_topics['topic_4'].sort_values(ascending=False)

@word  клетка             0.030503
       ген                0.012852
       организм           0.009684
       днк                0.009340
       пациент            0.009092
                            ...   
       камень             0.000000
       встревожить        0.000000
       технологический    0.000000
       летопись           0.000000
       аврелий            0.000000
Name: topic_4, Length: 19186, dtype: float64

In [595]:
bank_model.get_phi()['topic_4'].sort_values(ascending=False)

modality  token            
@word     язык                 0.015199
          книга                0.008724
          государство          0.007782
          вопрос               0.007237
          общество             0.004902
                                 ...   
          жорж                 0.000000
          пешеход              0.000000
          скучать              0.000000
          восстановительный    0.000000
          аврелий              0.000000
Name: topic_4, Length: 19186, dtype: float32

In [596]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9  # 10 ** 9

    def __init__(self, name: str, phi, topic_names: List[str]):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._phi = phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)
        parent_phi = self._phi
        
        rwt[:, self._topic_indices] += parent_phi.values[:, self._topic_indices]

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [600]:
bank_model._fit(
    optimizer._dataset.get_batch_vectorizer(),
    num_iterations=5,
    custom_regularizers={
        'fix': FastFixPhiRegularizer(
            name='fix',
            phi=bank_phi,
            topic_names=bank_phi.columns,
        ),
    }
)

In [601]:
bank_model.get_phi()['topic_4'].sort_values(ascending=False)

modality  token          
@word     клетка             0.030503
          ген                0.012852
          организм           0.009684
          днк                0.009340
          пациент            0.009092
                               ...   
          камень             0.000000
          встревожить        0.000000
          технологический    0.000000
          летопись           0.000000
          аврелий            0.000000
Name: topic_4, Length: 19186, dtype: float32

In [602]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', bank_model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', bank_model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', bank_model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', bank_model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', bank_model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@snippet [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@snippet_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@post_tag [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@post_tag_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@word [5297.6376953125, 5158.78857421875, 4709.48095703125, 4687.263671875, 4683.79833984375, 4682.92138671875, 4682.62109375, 4682.52197265625, 4682.48974609375, 4682.478515625]
test_ppl@word_with_d [5297.6376953125, 5158.78857421875, 4709.48095703125, 4687.263671875, 4683.79833984

In [475]:
bank_model.scores['perplexity_from_start']

KeyError: 'perplexity_from_start'

In [ ]:
DATASET.get_possible_modalities()

In [191]:
bank_model.get_phi()['topic_2'].sort_values(ascending=False)

modality  token         
@word     звезда            0.021522
          вселенная         0.016877
          галактика         0.016023
          земля             0.010558
          солнце            0.009292
                              ...   
          марш              0.000000
          феминистка        0.000000
          возобновляться    0.000000
          обвал             0.000000
          акт               0.000000
Name: topic_2, Length: 19186, dtype: float32

In [192]:
coherence_score.call_by_topic(bank_model)

{0: array([1.30200985]),
 1: array([1.08845739]),
 2: array([1.0168754]),
 3: array([1.15621355]),
 4: array([1.28988277])}

In [193]:
coherence_score.call(bank_model)

array([1.17068779])

In [194]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 2707.664306640625,
 'coherence_20': 1.1706877915144434,
 'diversity_euclidean': 0.06451756106182875,
 'diversity_jensenshannon': 0.6715327504064806,
 'diversity_hellinger': 0.7877305942486647,
 'diversity_cosine': 0.8494073511502591,
 'perplexity': 2707.664306640625}

In [195]:
optimizer._get_default_scores(bank_model)

{'perplexity_score': 2707.679443359375,
 'coherence_20': 1.1706877915144434,
 'diversity_euclidean': 0.0645175614263375,
 'diversity_jensenshannon': 0.6715327515781992,
 'diversity_hellinger': 0.7877305960342527,
 'diversity_cosine': 0.8494073528332182,
 'perplexity': 2707.679443359375}

In [196]:
bank_topics.shape

(19186, 5)

In [416]:
model = init_model_from_family('PLSA', DATASET, MAIN_MODALITY, 5, 0)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [417]:
model.get_phi().shape

(19186, 5)

In [418]:
for m in DATASET.get_possible_modalities():
    model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}',
            class_ids=[m]
        )
    )
    model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}_with_d',
            class_ids=[m],
            dictionary=DATASET.get_dictionary()
        )
    )

model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all',
        class_ids=DATASET.get_possible_modalities(),
    )
)
model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2',
   )
)
model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2_with_d',
        dictionary=DATASET.get_dictionary()
   )
)

In [419]:
model._fit(DATASET.get_batch_vectorizer(), 20)

In [420]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@snippet [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@snippet_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@post_tag [nan, nan, nan, na

In [421]:
model._fit(
    DATASET.get_batch_vectorizer(),
    num_iterations=20,
    custom_regularizers={
        'fix': FastFixPhiRegularizer(
            name='fix',
            phi=bank_phi,
            topic_names=bank_phi.columns,
        ),
    }
)

In [422]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan

In [412]:
bank_phi['topic_2'].sort_values(ascending=False)

@word  звезда            0.021526
       вселенная         0.016880
       галактика         0.016026
       земля             0.010559
       солнце            0.009293
                           ...   
       марш              0.000000
       феминистка        0.000000
       возобновляться    0.000000
       обвал             0.000000
       акт               0.000000
Name: topic_2, Length: 19186, dtype: float64

In [413]:
model.get_phi()['topic_2'].sort_values(ascending=False)

modality  token         
@word     звезда            0.021526
          вселенная         0.016880
          галактика         0.016026
          земля             0.010559
          солнце            0.009293
                              ...   
          марш              0.000000
          феминистка        0.000000
          возобновляться    0.000000
          обвал             0.000000
          акт               0.000000
Name: topic_2, Length: 19186, dtype: float32

In [415]:
model.get_phi()['background_5'].sort_values(ascending=False)

modality  token     
@word     говорить      0.002969
          язык          0.002913
          книга         0.002437
          большой       0.002309
          слово         0.002295
                          ...   
          шриффер       0.000007
          кваркова      0.000007
          килопарсек    0.000007
          оннести       0.000007
          камерлинг     0.000006
Name: background_5, Length: 19186, dtype: float32

In [384]:
model._fit(
    DATASET.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        'fix': FastFixPhiRegularizer(
            name='fix',
            phi=bank_phi,
            topic_names=bank_phi.columns,
        ),
    }
)

In [385]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan

In [386]:
bank_phi['topic_2'].sort_values(ascending=False)

@word  звезда            0.021526
       вселенная         0.016880
       галактика         0.016026
       земля             0.010559
       солнце            0.009293
                           ...   
       марш              0.000000
       феминистка        0.000000
       возобновляться    0.000000
       обвал             0.000000
       акт               0.000000
Name: topic_2, Length: 19186, dtype: float64

In [387]:
model.get_phi()['topic_2'].sort_values(ascending=False)

modality  token        
@word     звезда           0.018555
          вселенная        0.014553
          галактика        0.013789
          земля            0.009610
          объект           0.008414
                             ...   
          схватка          0.000000
          стратегически    0.000000
          вырабатывать     0.000000
          стул             0.000000
          акт              0.000000
Name: topic_2, Length: 19186, dtype: float32

In [113]:
model.scores['PerplexityScore@word']

[19895.9453125,
 5456.2392578125,
 5297.1337890625,
 5065.22119140625,
 4837.330078125,
 4680.2939453125,
 4590.2880859375,
 4540.271484375,
 4511.2646484375,
 4493.751953125,
 4482.685546875,
 4475.7783203125,
 4471.40087890625,
 4468.5302734375,
 4466.552734375,
 4465.248046875,
 4464.421875,
 4463.90576171875,
 4463.64404296875,
 4463.5302734375,
 4463.5283203125,
 5405.4609375,
 4719.11279296875,
 4688.26416015625,
 4683.736328125,
 4682.79052734375,
 4682.53857421875,
 4682.48193359375,
 4682.46923828125,
 4682.4638671875,
 4682.4619140625,
 4682.4658203125,
 4682.46875,
 4682.46875,
 4682.47021484375,
 4682.4697265625,
 4682.470703125,
 4682.470703125,
 4682.47021484375,
 4682.470703125,
 4682.470703125,
 4682.47119140625,
 4682.47119140625,
 4682.47119140625]

In [108]:
bank_model.get_phi()['topic_0'].sort_values(ascending=False)[:10]

modality  token            
@word     caption              0.025622
          align                0.012789
          width                0.012788
          attachment           0.012606
          свет                 0.012437
          aligncenter          0.007243
          рис                  0.006824
          устройство           0.006685
          сверхпроводимость    0.005892
          святилище            0.005607
Name: topic_0, dtype: float32

In [114]:
model.get_phi()['topic_0'].sort_values(ascending=False)[:10]

modality  token            
@word     caption              0.025624
          align                0.012790
          width                0.012790
          attachment           0.012607
          свет                 0.012438
          aligncenter          0.007243
          рис                  0.006825
          устройство           0.006685
          сверхпроводимость    0.005892
          святилище            0.005608
Name: topic_0, dtype: float32

In [115]:
model.get_phi().head()

topic_0  topic_1  topic_2   topic_3  topic_4  \
modality token                                                        
@word    инвалидность      0.0      0.0      0.0  0.000000      0.0   
         мазка             0.0      0.0      0.0  0.000000      0.0   
         professor         0.0      0.0      0.0  0.000017      0.0   
         умно              0.0      0.0      0.0  0.000000      0.0   
         игил              0.0      0.0      0.0  0.000000      0.0   

                       background_5  
modality token                       
@word    инвалидность      0.000010  
         мазка             0.000010  
         professor         0.000009  
         умно              0.000010  
         игил              0.000010

In [116]:
model = init_model_from_family('PLSA', DATASET, MAIN_MODALITY, 5, 0)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [117]:
model._fit(
    DATASET.get_batch_vectorizer(),
    num_iterations=20,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

In [118]:
model.scores['PerplexityScore@word']

[18982.515625,
 24645.68359375,
 23630.40234375,
 22722.39453125,
 22194.84765625,
 21943.359375,
 21827.154296875,
 21770.6875,
 21740.708984375,
 21723.21875,
 21712.1875,
 21704.763671875,
 21699.482421875,
 21695.611328125,
 21692.720703125,
 21690.517578125,
 21688.822265625,
 21687.4921875,
 21686.439453125,
 21685.58984375]

In [130]:
model.scores['PerplexityScore@all']

[18982.515625,
 24645.68359375,
 23630.40234375,
 22722.39453125,
 22194.84765625,
 21943.359375,
 21827.154296875,
 21770.6875,
 21740.708984375,
 21723.21875,
 21712.1875,
 21704.763671875,
 21699.482421875,
 21695.611328125,
 21692.720703125,
 21690.517578125,
 21688.822265625,
 21687.4921875,
 21686.439453125,
 21685.58984375]

In [125]:
bank_model.get_phi()['topic_1'].sort_values(ascending=False)[:10]

modality  token    
@word     право        0.019665
          закон        0.009846
          сталин       0.008532
          власть       0.007165
          история      0.006088
          суд          0.005563
          церковь      0.005529
          бог          0.005110
          отношение    0.004377
          должный      0.004363
Name: topic_1, dtype: float32

In [126]:
model.get_phi()['topic_1'].sort_values(ascending=False)[:10]

modality  token    
@word     право        0.019660
          закон        0.009844
          сталин       0.008530
          власть       0.007163
          история      0.006086
          суд          0.005561
          церковь      0.005527
          бог          0.005108
          отношение    0.004376
          должный      0.004363
Name: topic_1, dtype: float32

In [121]:
model.get_phi().head()

topic_0       topic_1       topic_2       topic_3  \
modality token                                                                  
@word    инвалидность  0.000000e+00  4.833264e-09  0.000000e+00  0.000000e+00   
         мазка         0.000000e+00  2.413737e-09  1.678086e-09  5.836904e-16   
         professor     0.000000e+00  0.000000e+00  0.000000e+00  1.689069e-05   
         умно          1.873608e-14  4.886941e-09  0.000000e+00  1.101553e-10   
         игил          0.000000e+00  4.997070e-09  0.000000e+00  0.000000e+00   

                            topic_4  
modality token                       
@word    инвалидность  1.638571e-10  
         мазка         9.061746e-10  
         professor     0.000000e+00  
         умно          0.000000e+00  
         игил          0.000000e+00

In [123]:
bank_model.get_phi().index

MultiIndex([('@word',   'инвалидность'),
            ('@word',          'мазка'),
            ('@word',      'professor'),
            ('@word',           'умно'),
            ('@word',           'игил'),
            ('@word',        'стучать'),
            ('@word',   'медлительный'),
            ('@word',    'газопылевой'),
            ('@word',       'разогрев'),
            ('@word',       'загрузка'),
            ...
            ('@word',           'зять'),
            ('@word', 'ниспровержение'),
            ('@word',        'пометка'),
            ('@word',       'франциск'),
            ('@word',           'wall'),
            ('@word',     'милосердие'),
            ('@word',        'поверка'),
            ('@word',            'вто'),
            ('@word',         'слоить'),
            ('@word',            'акт')],
           names=['modality', 'token'], length=19186)

In [122]:
model.get_phi().index

MultiIndex([('@word',   'инвалидность'),
            ('@word',          'мазка'),
            ('@word',      'professor'),
            ('@word',           'умно'),
            ('@word',           'игил'),
            ('@word',        'стучать'),
            ('@word',   'медлительный'),
            ('@word',    'газопылевой'),
            ('@word',       'разогрев'),
            ('@word',       'загрузка'),
            ...
            ('@word',           'зять'),
            ('@word', 'ниспровержение'),
            ('@word',        'пометка'),
            ('@word',       'франциск'),
            ('@word',           'wall'),
            ('@word',     'милосердие'),
            ('@word',        'поверка'),
            ('@word',            'вто'),
            ('@word',         'слоить'),
            ('@word',            'акт')],
           names=['modality', 'token'], length=19186)

In [173]:
all(bank_model.get_phi().index == model.get_phi().index)

True

In [124]:
optimizer._get_default_scores(model)

KeyError: 'perplexity_score'

In [144]:
bank_model.scores['perplexity_score']

[1836.541748046875,
 2707.664306640625,
 2707.67919921875,
 2707.679443359375,
 2707.679443359375,
 2707.679443359375,
 2707.679443359375,
 2707.679443359375]

In [147]:
bank_model.scores[f'PerplexityScore{modality}']

[nan, nan, nan, nan, nan, nan, nan]

In [150]:
bank_model.scores[f'test_ppl']

[2707.679443359375,
 2707.679443359375,
 2707.679443359375,
 2707.679443359375,
 2707.679443359375]

In [134]:
np.allclose(model.get_phi().values, bank_model.get_phi().values, atol=1e-5)

True

In [157]:
# model._model.scores.add(
#     artm.scores.PerplexityScore(name='test_ppl', class_ids=None)
# )
# model._model.scores.add(
#     artm.scores.PerplexityScore(name='test_ppl2', class_ids=None, dictionary=DATASET.get_dictionary())
# )
model._model.scores.add(
    artm.scores.PerplexityScore(name='test_ppl3', dictionary=DATASET.get_dictionary())
)
model._fit(
    DATASET.get_batch_vectorizer(),
    num_iterations=5,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

In [174]:
for m in DATASET.get_possible_modalities():
    model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}',
            class_ids=[m]
        )
    )
    model._model.scores.add(
        artm.scores.PerplexityScore(
            name=f'test_ppl{m}_with_d',
            class_ids=[m],
            dictionary=DATASET.get_dictionary()
        )
    )

model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all',
        class_ids=DATASET.get_possible_modalities(),
    )
)
model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2',
   )
)
model._model.scores.add(
    artm.scores.PerplexityScore(
        name=f'test_ppl@all_2_with_d',
        dictionary=DATASET.get_dictionary()
   )
)

In [175]:
model._fit(
    DATASET.get_batch_vectorizer(),
    num_iterations=5,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

In [143]:
model.scores['test_ppl']

[21684.888671875,
 21684.30078125,
 21683.7890625,
 21683.34375,
 21682.94921875,
 21682.603515625,
 21682.29296875,
 21682.017578125,
 21681.7734375,
 21681.55859375,
 21681.365234375,
 21681.1953125,
 21681.041015625,
 21680.896484375,
 21680.759765625,
 21680.634765625,
 21680.515625,
 21680.396484375,
 21680.28125,
 21680.1640625]

In [156]:
model.scores['test_ppl2']

[21680.048828125,
 21679.92578125,
 21679.798828125,
 21679.6640625,
 21679.521484375,
 21679.37109375,
 21679.2109375,
 21679.03515625,
 21678.841796875,
 21678.62109375,
 21678.357421875,
 21678.060546875,
 21677.73046875,
 21677.384765625,
 21677.048828125,
 21676.75390625,
 21676.513671875,
 21676.330078125,
 21676.201171875,
 21676.1015625]

In [158]:
model.scores['test_ppl3']

[21676.0234375, 21675.96484375, 21675.912109375, 21675.875, 21675.837890625]

In [154]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', bank_model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', bank_model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', bank_model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', bank_model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', bank_model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan]
test_ppl@author_with_d [nan, nan, nan, nan, nan]
test_ppl@snippet [nan, nan, nan, nan, nan]
test_ppl@snippet_with_d [nan, nan, nan, nan, nan]
test_ppl@post_tag [nan, nan, nan, nan, nan]
test_ppl@post_tag_with_d [nan, nan, nan, nan, nan]
test_ppl@word [nan, nan, nan, nan, nan]
test_ppl@word_with_d [nan, nan, nan, nan, nan]
test_ppl@3gramm [nan, nan, nan, nan, nan]
test_ppl@3gramm_with_d [nan, nan, nan, nan, nan]
test_ppl@all [nan, nan, nan, nan, nan]
test_ppl@all_2 [2707.679443359375, 2707.679443359375, 2707.679443359375, 2707.679443359375, 2707.679443359375]
test_ppl@all_2_with_d [6035.37841796875, 6035.37841796875, 6035.37841796875, 6035.37841796875, 6035.37841796875]


In [ ]:
# WTF???? all_2 vs all_2_with_d

In [176]:
for m in DATASET.get_possible_modalities():
    print(f'test_ppl{m}', model.scores[f'test_ppl{m}'])
    print(f'test_ppl{m}_with_d', model.scores[f'test_ppl{m}_with_d'])


print(f'test_ppl@all', model.scores[f'test_ppl@all'])
print(f'test_ppl@all_2', model.scores[f'test_ppl@all_2'])
print(f'test_ppl@all_2_with_d', model.scores[f'test_ppl@all_2_with_d'])

test_ppl@title [nan, nan, nan, nan, nan]
test_ppl@title_with_d [nan, nan, nan, nan, nan]
test_ppl@2gramm [nan, nan, nan, nan, nan]
test_ppl@2gramm_with_d [nan, nan, nan, nan, nan]
test_ppl@author [nan, nan, nan, nan, nan]
test_ppl@author_with_d [nan, nan, nan, nan, nan]
test_ppl@snippet [nan, nan, nan, nan, nan]
test_ppl@snippet_with_d [nan, nan, nan, nan, nan]
test_ppl@post_tag [nan, nan, nan, nan, nan]
test_ppl@post_tag_with_d [nan, nan, nan, nan, nan]
test_ppl@word [21675.8046875, 21675.7734375, 21675.744140625, 21675.7109375, 21675.67578125]
test_ppl@word_with_d [21675.8046875, 21675.7734375, 21675.744140625, 21675.7109375, 21675.67578125]
test_ppl@3gramm [nan, nan, nan, nan, nan]
test_ppl@3gramm_with_d [nan, nan, nan, nan, nan]
test_ppl@all [21675.8046875, 21675.7734375, 21675.744140625, 21675.7109375, 21675.67578125]
test_ppl@all_2 [21675.8046875, 21675.7734375, 21675.744140625, 21675.7109375, 21675.67578125]
test_ppl@all_2_with_d [21675.8046875, 21675.7734375, 21675.744140625, 2

In [159]:
np.allclose(model.get_phi().values, bank_model.get_phi().values, atol=1e-5)

True

In [163]:
np.allclose(model.get_theta(dataset=DATASET).values, bank_model.get_theta(dataset=DATASET).values, atol=1e-2)

False

In [166]:
model.get_theta(dataset=DATASET).head()

,2001.txt,2002.txt,2003.txt,2004.txt,2005.txt,2006.txt,2007.txt,2008.txt,2009.txt,2010.txt,...,991.txt,992.txt,993.txt,994.txt,995.txt,996.txt,997.txt,998.txt,999.txt,1000.txt
topic_0,0.056126,0.051794,0.027882,0.014199,0.043396,0.055867,0.077195,0.026650,0.108815,0.070042,...,0.046440,0.093659,0.037586,0.064593,0.000370,0.101226,0.006319,0.067003,0.146280,0.014262
topic_1,0.450934,0.030464,0.715602,0.750854,0.780230,0.683530,0.045779,0.174325,0.094343,0.779998,...,0.304143,0.398997,0.493840,0.040556,0.026051,0.104955,0.090535,0.459343,0.388860,0.046970
topic_2,0.456754,0.081020,0.108327,0.098719,0.082993,0.080848,0.186471,0.160910,0.130105,0.052453,...,0.252047,0.008952,0.303512,0.072870,0.883414,0.141408,0.106314,0.130316,0.053466,0.427126
topic_3,0.001702,0.121700,0.009623,0.045128,0.014734,0.064188,0.183776,0.140578,0.059057,0.026539,...,0.050335,0.154085,0.091995,0.060165,0.090149,0.129368,0.087532,0.112709,0.094390,0.501772
topic_4,0.034484,0.715021,0.138565,0.091100,0.078646,0.115567,0.506779,0.497538,0.607680,0.070969,...,0.347036,0.344307,0.073068,0.761816,0.000015,0.523044,0.709301,0.230629,0.317003,0.009870


In [168]:
model._cache_theta

False

In [170]:
model._model.transform(DATASET.get_batch_vectorizer())

,2001.txt,2002.txt,2003.txt,2004.txt,2005.txt,2006.txt,2007.txt,2008.txt,2009.txt,2010.txt,...,991.txt,992.txt,993.txt,994.txt,995.txt,996.txt,997.txt,998.txt,999.txt,1000.txt
topic_0,0.056126,0.051794,0.027882,0.014199,0.043396,0.055867,0.077195,0.026650,0.108815,0.070042,...,0.046440,0.093659,0.037586,0.064593,0.000370,0.101226,0.006319,0.067003,0.146280,0.014262
topic_1,0.450934,0.030464,0.715602,0.750854,0.780230,0.683530,0.045779,0.174325,0.094343,0.779998,...,0.304143,0.398997,0.493840,0.040556,0.026051,0.104955,0.090535,0.459343,0.388860,0.046970
topic_2,0.456754,0.081020,0.108327,0.098719,0.082993,0.080848,0.186471,0.160910,0.130105,0.052453,...,0.252047,0.008952,0.303512,0.072870,0.883414,0.141408,0.106314,0.130316,0.053466,0.427126
topic_3,0.001702,0.121700,0.009623,0.045128,0.014734,0.064188,0.183776,0.140578,0.059057,0.026539,...,0.050335,0.154085,0.091995,0.060165,0.090149,0.129368,0.087532,0.112709,0.094390,0.501772
topic_4,0.034484,0.715021,0.138565,0.091100,0.078646,0.115567,0.506779,0.497538,0.607680,0.070969,...,0.347036,0.344307,0.073068,0.761816,0.000015,0.523044,0.709301,0.230629,0.317003,0.009870


In [167]:
bank_model.get_theta(dataset=DATASET).head()

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,990,991,992,993,994,995,996,997,998,999
topic_0,0.101861,0.074489,0.049455,0.022172,0.054905,0.070409,0.101073,0.037551,0.134165,0.113353,...,0.070619,0.112992,0.051316,0.073642,0.000986,0.127711,0.011525,0.099503,0.197428,0.015720
topic_1,0.655863,0.000609,0.372001,0.575786,0.645551,0.599838,0.032964,0.064134,0.050619,0.640101,...,0.185637,0.275330,0.352115,0.023438,0.018515,0.040171,0.048147,0.304867,0.152107,0.015568
topic_2,0.055471,0.127861,0.242632,0.168248,0.112799,0.103563,0.227997,0.224257,0.158512,0.091768,...,0.374002,0.016707,0.380625,0.084925,0.889346,0.168104,0.144765,0.185562,0.086654,0.441052
topic_3,0.027465,0.175088,0.037685,0.078449,0.029023,0.080083,0.235881,0.200229,0.075055,0.041916,...,0.081005,0.197989,0.120509,0.075768,0.091138,0.174744,0.105661,0.166301,0.153787,0.515394
topic_4,0.159340,0.621953,0.298227,0.155344,0.157723,0.146106,0.402084,0.473828,0.581650,0.112861,...,0.288736,0.396982,0.095435,0.742226,0.000015,0.489270,0.689901,0.243766,0.410024,0.012266


In [169]:
bank_model._cache_theta

False

In [171]:
bank_model._model.transform(DATASET.get_batch_vectorizer())

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,990,991,992,993,994,995,996,997,998,999
topic_0,0.101861,0.074489,0.049455,0.022172,0.054905,0.070409,0.101073,0.037551,0.134165,0.113353,...,0.070619,0.112992,0.051316,0.073642,0.000986,0.127711,0.011525,0.099503,0.197428,0.015720
topic_1,0.655863,0.000609,0.372001,0.575786,0.645551,0.599838,0.032964,0.064134,0.050619,0.640101,...,0.185637,0.275330,0.352115,0.023438,0.018515,0.040171,0.048147,0.304867,0.152107,0.015568
topic_2,0.055471,0.127861,0.242632,0.168248,0.112799,0.103563,0.227997,0.224257,0.158512,0.091768,...,0.374002,0.016707,0.380625,0.084925,0.889346,0.168104,0.144765,0.185562,0.086654,0.441052
topic_3,0.027465,0.175088,0.037685,0.078449,0.029023,0.080083,0.235881,0.200229,0.075055,0.041916,...,0.081005,0.197989,0.120509,0.075768,0.091138,0.174744,0.105661,0.166301,0.153787,0.515394
topic_4,0.159340,0.621953,0.298227,0.155344,0.157723,0.146106,0.402084,0.473828,0.581650,0.112861,...,0.288736,0.396982,0.095435,0.742226,0.000015,0.489270,0.689901,0.243766,0.410024,0.012266
